# 01 — Análisis exploratorio (EDA)

Exploración del precio eléctrico diario y covariables (demanda, embalse, aportes, ENSO).


In [ ]:
import sys
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(12):
        if (p / "src" / "xm_config.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(
        "Abra Jupyter con cwd en la raíz del repo o en notebooks/ "
        "(debe existir src/xm_config.py)."
    )

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from eda_analysis import load_cache_only, load_daily_auto, resolve_daily_price_target, missing_summary, numeric_columns_for_corr
from xm_config import TARGET_COL_DAILY_WEIGHTED_PRICE

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.titlesize"] = 11

try:
    df = load_cache_only()
    print("Datos: caché local")
except FileNotFoundError:
    df = load_daily_auto(date(2015, 1, 1), date(2025, 12, 31))
    print("Datos: descarga / construcción automática")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

TARGET = resolve_daily_price_target(df, prefer_weighted=True)
print("Filas:", len(df), "| Desde", df["date"].min().date(), "hasta", df["date"].max().date())
print("Columna objetivo (EDA):", TARGET)
if TARGET != TARGET_COL_DAILY_WEIGHTED_PRICE:
    print(
        "ADVERTENCIA: no se usa la columna ponderada. Verifique xm_daily_max_price_dataset.parquet "
        "y que exista precio_prom_ponderado_cop_kwh con valores."
    )
df.head()


In [ ]:
display(df.describe().T)
display(missing_summary(df))


## 1. Estacionariedad (ADF y KPSS)

Contraste de raíz unitaria (ADF) y estacionariedad en media (KPSS) sobre el precio en niveles.


In [ ]:
import warnings

from statsmodels.tsa.stattools import adfuller, kpss

y_stat = pd.to_numeric(df[TARGET], errors="coerce").dropna()
if len(y_stat) < 50:
    print("Serie demasiado corta para ADF/KPSS.")
else:
    adf_out = adfuller(y_stat.values, autolag="AIC")
    adf_stat, adf_p = float(adf_out[0]), float(adf_out[1])
    adf_usedlag = int(adf_out[2])
    adf_crit = adf_out[4] if len(adf_out) > 4 else {}
    print("--- ADF (Dickey–Fuller aumentado) — niveles ---")
    print(f"  Estadístico: {adf_stat:.6g}")
    print(f"  p-valor:     {adf_p:.6g}")
    print(f"  rezagos:     {adf_usedlag}")
    if isinstance(adf_crit, dict):
        print("  Criterios (1%, 5%, 10%):", {k: round(float(v), 4) for k, v in adf_crit.items()})
    print("  H0: existe raíz unitaria (no estacionaria). Rechazar H0 si p < α (serie estacionaria).")

    kpss_p = np.nan
    kpss_stat = np.nan
    kpss_lags = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(
                y_stat.values, regression="c", nlags="auto"
            )
        print("\n--- KPSS — niveles (constante) ---")
        print(f"  Estadístico: {kpss_stat:.6g}")
        print(f"  p-valor:     {kpss_p:.6g}")
        print("  H0: estacionariedad en torno a una constante. Rechazar H0 si p < α (evidencia de no estacionariedad).")
    except Exception as exc:
        print("\n--- KPSS ---")
        print("  No se pudo calcular:", exc)

    # Diferencia primera (log1p del precio, luego diff) — referencia para modelado
    z = np.log1p(np.maximum(y_stat.values.astype(float), 0.0))
    dz = np.diff(z)
    if len(dz) >= 50:
        adf_d = adfuller(dz, autolag="AIC")
        print("\n--- ADF sobre Δ log1p(precio) ---")
        print(f"  p-valor ADF: {adf_d[1]:.6g} (referencia para integración / modelos en diferencias)")


## 2. Distribución del precio

Histograma y estadísticos de forma: asimetría y curtosis.


In [ ]:
y_dist = pd.to_numeric(df[TARGET], errors="coerce").dropna()
sk = float(y_dist.skew())
ku = float(y_dist.kurtosis())  # exceso de curtosis (Fisher; normal → 0)

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(y_dist, kde=True, stat="density", ax=ax, color="steelblue", edgecolor="white")
ax.axvline(float(y_dist.mean()), color="darkred", ls="--", lw=1.2, label=f"Media = {y_dist.mean():.2f}")
ax.axvline(float(y_dist.median()), color="darkgreen", ls=":", lw=1.2, label=f"Mediana = {y_dist.median():.2f}")
ax.set_xlabel("COP/kWh")
ax.set_ylabel("Densidad")
ax.set_title(f"Distribución empírica — {TARGET}")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Asimetría (skewness): {sk:.4f}")
print(f"Curtosis (exceso, pandas): {ku:.4f}  (>0 → colas más pesadas que la normal)")


## 3. Volatilidad y heterocedasticidad

Rendimientos logarítmicos y volatilidad móvil. Motiva el uso de modelos GARCH en el notebook de pronóstico.


In [ ]:
td = pd.to_datetime(df["date"])
px = pd.to_numeric(df[TARGET], errors="coerce").clip(lower=0.0)
s_log = pd.Series(np.log1p(px.to_numpy(dtype=float)), index=td)
r = s_log.diff().dropna()
win = 30
roll_sig = r.rolling(win, min_periods=win).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(r.index, r.values, lw=0.55, color="C0", alpha=0.85)
axes[0].set_ylabel("Retorno log")
axes[0].set_title(f"Rendimientos diarios (log1p) — {TARGET}")

axes[1].plot(roll_sig.index, roll_sig.values, lw=1.0, color="darkred")
axes[1].set_ylabel(f"σ rolling ({win}d)")
axes[1].set_xlabel("Fecha")
axes[1].set_title("Variabilidad local (rolling): heterocedasticidad / clustering de volatilidad")
plt.tight_layout()
plt.show()

if len(r) > 40:
    from statsmodels.graphics.tsaplots import plot_acf

    fig2, ax2 = plt.subplots(figsize=(10, 3.2))
    plot_acf(np.asarray(r, dtype=float) ** 2, ax=ax2, lags=40, title="ACF de rendimientos al cuadrado (proxy ARCH)")
    plt.tight_layout()
    plt.show()


## 4. Series en el tiempo

Una figura por variable del panel diario. Escala propia en cada gráfico (COP/kWh, kWh, proporciones, índice ENSO).

| Variable | Descripción breve |
|----------|-----------------|
| `precio_prom_ponderado_cop_kwh` | Precio de bolsa, promedio ponderado por demanda (COP/kWh) |
| `demanda_max_kwh` | Máximo horario de demanda comercial |
| `porc_vol_util_diario` | Volumen útil del sistema hídrico (fracción) |
| `porc_aporte_value` | Aporte hídrico medio diario |
| `enso_index` | Índice ENSO (covariable climática) |


In [ ]:
import matplotlib.dates as mdates

t = pd.to_datetime(df["date"])
cols_ts = [
    c
    for c in [
        TARGET,
        "demanda_max_kwh",
        "porc_vol_util_diario",
        "vol_util_energia_sistema",
        "porc_aporte_value",
        "porc_aporte_max",
        "enso_index",
    ]
    if c in df.columns
]
if not cols_ts:
    raise ValueError("No hay columnas de series temporales para graficar.")

# --- Frecuencia / calendario (para el texto del documento) ---
dt_idx = pd.DatetimeIndex(t)
gaps = t.diff().dropna()
print(
    "Rango:",
    t.min().date(),
    "→",
    t.max().date(),
    "| filas:",
    len(df),
    "| columnas a graficar:",
    len(cols_ts),
)
if len(gaps):
    print("Δt entre filas consecutivas — mediana:", gaps.median(), "| moda:", gaps.mode(dropna=True).iloc[0] if len(gaps.mode(dropna=True)) else "—")
print("infer_freq (pandas):", pd.infer_freq(dt_idx) or "(no inferido; posibles huecos o fechas irregulares)")

# --- Estadísticos descriptivos (mismas series que los gráficos) ---
ts_num = df[cols_ts].apply(pd.to_numeric, errors="coerce")
display(ts_num.describe().T)

_TS_LABELS: dict[str, tuple[str, str]] = {
    "precio_prom_ponderado_cop_kwh": (
        "Precio bolsa",
        "COP / kWh",
    ),
    "precio_max_cop_kwh": ("Precio bolsa (máximo horario del día)", "COP / kWh"),
    "demanda_max_kwh": ("Demanda comercial (máximo horario del día)", "kWh"),
    "porc_vol_util_diario": ("Volumen útil diario", "Porcentaje"),
    "vol_util_energia_sistema": ("Volumen útil de energía del sistema", "kWh"),
    "porc_aporte_value": ("Aporte de fuentes de energía", "Porcentaje"),
    "porc_aporte_max": ("% aporte (máximo horario del día)", "Porcentaje"),
    "enso_index": ("Índice ENSO", "adimensional"),
}


def _ts_title_ylabel(col: str) -> tuple[str, str]:
    if col in _TS_LABELS:
        return _TS_LABELS[col]
    return (col.replace("_", " "), col)


for col in cols_ts:
    title, ylab = _ts_title_ylabel(col)
    y = pd.to_numeric(df[col], errors="coerce")
    fig, ax = plt.subplots(figsize=(12, 3.8), constrained_layout=True)
    ax.plot(t, y, lw=0.9, color="C0")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylab, fontsize=10)
    ax.set_xlabel("Fecha", fontsize=10)
    ax.tick_params(axis="both", which="major", labelsize=9)
    ax.grid(True, which="major", alpha=0.35)
    # Eje X visible en cada figura: marcas por año
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.margins(x=0.01)
    ax.text(
        0.5,
        -0.22,
        col,
        transform=ax.transAxes,
        fontsize=8,
        color="0.35",
        ha="center",
        va="top",
        family="monospace",
        clip_on=False,
    )
    plt.show()


## 5. Estacionalidad y descomposición STL

- Ciclo medio mensual del precio (media ± error estándar).
- Descomposición STL (tendencia, estacionalidad, residuo) con período anual (365 días).


In [ ]:
from statsmodels.tsa.seasonal import STL

MON_LABELS = [
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
]

_STL_BLUE = "#1f77b4"  # azul único (estilo tab:blue) para toda esta sección


def _stl_yunit(col: str) -> str:
    """Unidad coherente con _TS_LABELS (celda de series en el tiempo)."""
    d = globals().get("_TS_LABELS")
    if isinstance(d, dict) and col in d:
        return d[col][1]
    return ""


# --- Fig. 1: nivel típico del precio por mes calendario (todos los años) ---
mcal = pd.to_datetime(df["date"]).dt.month
g = df.groupby(mcal)[TARGET]

mean_m = g.mean()
std_m = g.std(ddof=0)
cnt_m = g.count()
sem_m = std_m / np.sqrt(np.maximum(cnt_m, 1))

xmonth = np.arange(1, 13)
mu = mean_m.reindex(xmonth)

_u_target = _stl_yunit(TARGET) or "COP / kWh"
fig1, ax1 = plt.subplots(figsize=(10, 3.6))
ax1.plot(
    xmonth,
    mu,
    marker="o",
    ms=5,
    lw=1.8,
    color=_STL_BLUE,
    markerfacecolor=_STL_BLUE,
    markeredgecolor=_STL_BLUE,
    label="Promedio diario del mes",
)
band = sem_m.reindex(xmonth)
ax1.fill_between(
    xmonth,
    (mu - band).values.astype(float),
    (mu + band).values.astype(float),
    color=_STL_BLUE,
    alpha=0.22,
    label="± SEM (entre días dentro del mes)",
)
ax1.set_xticks(xmonth)
ax1.set_xticklabels(MON_LABELS)
ax1.set_xlabel("Mes calendario")
ax1.set_ylabel(_u_target)
ax1.legend(loc="best", fontsize=8)
ax1.set_title(f"Ciclo estacional medio (mes a mes) — {TARGET}")
plt.tight_layout()
plt.show()

# --- STL: todas las variables numéricas excepto lista de exclusión (p. ej. mes) ---
STL_PERIOD = 365  # intra-anual sobre serie diaria
MIN_LEN = 2 * STL_PERIOD + 5
SKIP_DECOMP = {"date", "mes", "enso_regime"}

dec_cols = [
    c
    for c in df.columns
    if c not in SKIP_DECOMP and pd.api.types.is_numeric_dtype(df[c])
]
if TARGET in dec_cols:
    dec_cols = [TARGET] + sorted(c for c in dec_cols if c != TARGET)
else:
    dec_cols = sorted(dec_cols)

idx = pd.DatetimeIndex(pd.to_datetime(df["date"])).rename("date")
base = df.set_index(idx)[dec_cols].sort_index()
base = base[~base.index.duplicated(keep="first")]

for col in dec_cols:
    s = base[col].astype(float).asfreq("D")
    s = s.interpolate(method="linear", limit=31, limit_direction="both")
    if not np.isfinite(s).all():
        s = s.replace([np.inf, -np.inf], np.nan).interpolate(method="linear", limit=31, limit_direction="both")
    s = s.dropna()
    if len(s) < MIN_LEN:
        print(f"[STL] Omito '{col}': n={len(s)} < {MIN_LEN}")
        continue
    if float(s.std(ddof=0)) < 1e-12:
        print(f"[STL] Omito '{col}': serie casi constante.")
        continue
    try:
        res = STL(s, period=STL_PERIOD, robust=True).fit()
    except Exception as exc:
        print(f"[STL] No aplicó a '{col}': {exc}")
        continue

    _u = _stl_yunit(col) or col
    fig_i, axes = plt.subplots(4, 1, figsize=(12, 6.8), sharex=True)
    axes[0].plot(s.index, s.values, lw=0.75, color=_STL_BLUE)
    axes[0].set_ylabel(f"Observ.\n({_u})")
    axes[1].plot(s.index, res.trend, lw=1.0, color=_STL_BLUE)
    axes[1].set_ylabel(f"Tendencia\n({_u})")
    axes[2].plot(s.index, res.seasonal, lw=0.65, color=_STL_BLUE)
    axes[2].set_ylabel(f"Estacional (m={STL_PERIOD})\n({_u})")
    axes[3].plot(s.index, res.resid, lw=0.45, alpha=0.9, color=_STL_BLUE)
    axes[3].set_ylabel(f"Residuo\n({_u})")
    axes[0].set_title(
        f"STL — {col}\n"
        f"Ciclo estacional anual (m={STL_PERIOD}) en calendario diario"
    )
    for ax in axes:
        ax.axhline(0.0, color="k", lw=0.35, alpha=0.35)
    axes[-1].set_xlabel("Fecha")
    plt.tight_layout()
    plt.show()


## 6. Medias móviles

In [ ]:
y = df[TARGET]
ma7 = y.rolling(7, min_periods=1).mean()
ma30 = y.rolling(30, min_periods=1).mean()
ma365 = y.rolling(365, min_periods=3).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["date"], y, alpha=0.35, label="Diario", lw=0.6)
ax.plot(df["date"], ma7, label="MA 7d", lw=1)
ax.plot(df["date"], ma30, label="MA 30d", lw=1.2)
ax.plot(df["date"], ma365, label="MA 365d", lw=1.5)
ax.set_ylabel("COP/kWh")
ax.legend(loc="upper left")
ax.set_title(f"Precio diario ({TARGET}) — medias móviles")
plt.tight_layout()
plt.show()


## 7. Correlación (Pearson)

In [ ]:
def _corr_heatmap_label(col: str) -> str:
    """Etiqueta legible para ejes del mapa de calor (coherente con _TS_LABELS si existe)."""
    d = globals().get("_TS_LABELS")
    if isinstance(d, dict) and col in d:
        title, unit = d[col]
        return f"{title}\n({unit})"
    extra = {
        "enso_index_lag1": "Índice ENSO\n(rezago 1 día)",
    }
    if col in extra:
        return extra[col]
    return col.replace("_", "\n")


num_cols = numeric_columns_for_corr(df)
corr = df[num_cols].corr(method="pearson")

_lbl_map = {c: _corr_heatmap_label(c) for c in corr.columns}
corr_plot = corr.rename(index=_lbl_map, columns=_lbl_map)

fig, ax = plt.subplots(figsize=(11.5, 10.2))
sns.heatmap(
    corr_plot,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    linewidths=0.35,
    linecolor="white",
    annot_kws={"size": 8},
)
ax.set_title("Correlación (Pearson)", pad=14)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.setp(
    ax.get_xticklabels(),
    rotation=48,
    ha="right",
    rotation_mode="anchor",
    fontsize=7.5,
)
fig.tight_layout(rect=(0, 0.22, 1, 0.98))
plt.show()

print("Correlación con el objetivo (|ρ| ordenado):")
s = corr[TARGET].drop(TARGET, errors="ignore")
order = s.abs().sort_values(ascending=False).index
s_show = s.reindex(order).rename(index=_lbl_map)
display(s_show.to_frame("ρ"))


## 8. Dispersión: precio vs covariables

In [ ]:
others = [c for c in num_cols if c != TARGET]
if not others:
    print("No hay covariables numéricas además del objetivo.")
else:
    n_o = min(len(others), 6)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    axes = axes.ravel()
    for ax, col in zip(axes, others[:n_o]):
        ax.scatter(df[col], df[TARGET], s=8, alpha=0.35)
        ax.set_xlabel(col, fontsize=8)
        ax.set_ylabel(TARGET, fontsize=8)
    for j in range(n_o, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f"Dispersión vs {TARGET}", y=1.02)
    plt.tight_layout()
    plt.show()


## 9. Precio por régimen ENSO

In [ ]:
if "enso_regime" in df.columns:
    sub = df.dropna(subset=["enso_regime"])
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=sub, x="enso_regime", y=TARGET, ax=ax, order=["la_nina", "neutral", "el_nino"])
    ax.set_title(f"{TARGET} por régimen ENSO")
    plt.tight_layout()
    plt.show()
else:
    print("No hay columna enso_regime.")


## 10. ACF y PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

y_clean = y.dropna()
if len(y_clean) > 40:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(y_clean, ax=axes[0], lags=40)
    plot_pacf(y_clean, ax=axes[1], lags=40, method="ywm")
    plt.suptitle(f"ACF / PACF — {TARGET}")
    plt.tight_layout()
    plt.show()
else:
    print("Serie demasiado corta para ACF/PACF.")
